In [7]:
import pandas as pd
import os
import joblib
from sklearn.preprocessing import StandardScaler

In [3]:
# -----------------------------
# Base paths
# -----------------------------
BASE_DIR = ".."
data_dir = os.path.join(BASE_DIR, "data")
models_dir = os.path.join(BASE_DIR, "models")

os.makedirs(data_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

# -----------------------------
# Input CSV files (from data folder)
# -----------------------------
csv_files = [
    os.path.join(data_dir, "1_airplanenew_features.csv"),
    os.path.join(data_dir, "5_cameraman_features.csv"),
    os.path.join(data_dir, "9_Splash_features.csv"),
]

# -----------------------------
# Load and combine datasets
# -----------------------------
df_list = [pd.read_csv(file) for file in csv_files]
df_combined = pd.concat(df_list, ignore_index=True)

# Shuffle dataset
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

# -----------------------------
# Save merged dataset
# -----------------------------
final_data_path = os.path.join(data_dir, "final_training_data.csv")
df_combined.to_csv(final_data_path, index=False)

print(f"Merged dataset saved at {final_data_path} with {len(df_combined)} samples") 

 Merged dataset saved as 'final_training_data.csv' with 3072 samples.


In [ ]:
# -----------------------------
# Labeling function
# -----------------------------
def assign_label(entropy):
    if entropy >= 4.364:
        return 1
    elif entropy >= 2.854:
        return 1
    else:
        return 0

# Apply labeling
df_combined['Encrypt'] = df_combined['Entropy'].apply(assign_label)

# -----------------------------
# Feature selection
# -----------------------------
features = ['Entropy', 'Energy', 'Correlation', 'Contrast', 'Homogeneity']
X = df_combined[features]
y = df_combined['Encrypt']

# -----------------------------
# Standardization
# -----------------------------
scaler = StandardScaler()
X_standardized = scaler.fit_transform(X)

# Save scaler
scaler_path = os.path.join(models_dir, "scaler.pkl")
joblib.dump(scaler, scaler_path)

print(f"Scaler saved at {scaler_path}")

# -----------------------------
# Save standardized dataset
# -----------------------------
df_standardized = pd.DataFrame(X_standardized, columns=features)
df_standardized['Encrypt'] = y

standardized_path = os.path.join(data_dir, "standardized_training_data.csv")
df_standardized.to_csv(standardized_path, index=False)

print(f"Standardized dataset saved at {standardized_path}")